##### 分期刊爬取 -- 爬虫问题失败

In [21]:
import pandas as pd
import os

files = os.listdir("data/all_journals")
df = pd.read_csv(f"data/all_journals/{files[0]}")
df = df.drop(columns=["Link"])

for file in files[1:]:
    df_temp = pd.read_csv(f"data/all_journals/{file}")
    df_temp = df_temp.drop(columns=["Link"])
    df = pd.concat([df, df_temp])

print(df.shape)

(2985, 7)


In [6]:
# 获取论文在Elsevier中对应的Pii号
import requests
import json
from tqdm import tqdm
url = 'https://api.elsevier.com/content/search/sciencedirect'
headers = {
    "X-ELS-APIKEY": '8d9aeaad617f95435bf6dc6d235594fb',
}

titles = df['Title'].tolist()
source_titles = df['Source title'].tolist()
piis = []
for i, title in enumerate(tqdm(titles)):
    data = {
        'title':title,
        'pub':f"\"{source_titles[i]}\"",   
    }

    response = requests.put(url, data=json.dumps(data), headers=headers)
    if response.status_code != 200:
        piis.append('error')
    elif response.json()['resultsFound'] == 0:
        piis.append('not found')
    else:
        piis.append(response.json()['results'][0]['pii'])

100%|██████████| 2985/2985 [10:38<00:00,  4.67it/s]


In [22]:
df['Pii'] = piis
df = df[~df['Pii'].isin(['not found', 'error'])]
df = df[['Title', 'Year', 'Source title', 'Abstract', 'Author Keywords', 'Pii']]
df = df[df['Abstract'] != '[No abstract available]']
print(df.shape)

(1169, 6)


In [24]:
df.to_excel("data/combine/all_expe.xlsx", index=False)

In [29]:
# 分析每个期刊对应的论文数，至少保留50条用于后续，不足50条的进行补充
import pandas as pd

df = pd.read_excel("data/combine/all_expe.xlsx")
piis = df['Pii'].tolist()
piis = ['https://www.sciencedirect.com/science/article/pii/'+pii for pii in piis]
df['Pii'] = piis
df.to_excel("data/combine/all_expe.xlsx", index=False)

source_titles = df['Source title'].tolist()
source_nums = {}
for i, title in enumerate(source_titles):
    if title in source_nums:
        source_nums[title] += 1
    else:
        source_nums[title] = 1

source_nums

{'Advances in Space Research': 43,
 'Annals of Emergency Medicine': 12,
 'Applied Mathematical Modelling': 40,
 'Artificial Intelligence': 49,
 'Brain, Behavior, and Immunity': 40,
 'Cell': 40,
 'Chem': 40,
 'Chemical Engineering Journal': 38,
 'Complementary Therapies in Medicine': 39,
 'Composites Science and Technology': 40,
 'Ecological Indicators': 45,
 'Ecosystem Services': 41,
 'Energy Storage Materials': 40,
 'EnergyChem': 51,
 'Human Factors in Healthcare': 37,
 'Innovation': 23,
 'Journal of Aging Studies': 57,
 'Journal of Economics and Business': 46,
 'Journal of Endodontics': 40,
 'Journal of Financial Stability': 43,
 'Journal of Theoretical Biology': 44,
 'Journal of Vocational Behavior': 63,
 'Life Sciences': 52,
 'Physics Reports': 47,
 'Social Networks': 62,
 'Trends in Chemistry': 54,
 'Veterinary Microbiology': 43}

##### Manual preprocessing -- with Doreen

In [1]:
import pandas as pd

df = pd.read_excel("data/combine/all_expe_by_self.xlsx")
df.head()

,Title,Abstract,Keywords,Highlights,Link
0,Advances in Space Research,This study explores the role of atmospheric wa...,Lapse rate;Precipitable water vapour;Scale hei...,Highlights\n•\nRadiative forcing by precipitab...,https://www.sciencedirect.com/science/article/...
1,Advances in Space Research,Water vapor is an essential component of the E...,Water vapor;FY-3G/MWRI-RM;ERA5;Radiosonde,Highlights\n•\nThis study uses the microwave o...,https://www.sciencedirect.com/science/article/...
2,Advances in Space Research,"Soil moisture, pivotal for water, energy, and ...",DBN;Soil moisture;Downscaling;Spatial resoluti...,Highlights\n•\nThe Deep Belief Network (DBN) i...,https://www.sciencedirect.com/science/article/...
3,Advances in Space Research,"Foldable membrane structures, with light weigh...",Space membrane structure;Crease dynamics;Dynam...,Highlights\n•\nIntegrated crease-contact model...,https://www.sciencedirect.com/science/article/...
4,Advances in Space Research,Equatorial plasma bubbles (EPBs) are ionospher...,Equatorial plasma bubbles;Ionospheric irregula...,Highlights\n•\nGlobal pre-midnight EPB pattern...,https://www.sciencedirect.com/science/article/...


In [2]:
abstracts = df['Abstract'].tolist()
highlights = df['Highlights'].tolist()
keywords = df['Keywords'].tolist()

In [3]:
import re

new_abstracts = []
for text in abstracts:
    text = text.strip().lower().replace('abstract','')
    text = re.sub(r'\s\([^)]*\)\s', '', text)
    new_abstracts.append(text.strip())

print(abstracts[:5])
print(new_abstracts[:5])

['This study explores the role of atmospheric water vapour as a climate forcing agent separately in three distinct forms, namely precipitable water vapour (PWV), scale height and moist lapse rate over the tropical regions of India utilizing radiosonde and reanalysis data. It demonstrates several significant findings, which contribute to the understanding of regional climate dynamics. Analysing the spatiotemporal variations of PWV over a 20-year period (2001–2021), an enhancement of ≈0.842–3.858 × 1019 J of heat energy trapped in the atmosphere is inferred, which corresponds to the radiative forcing of ≈0.0055–0.025 Wm−2 per year. The daily PWV values for four metropolitan cities of India show the annual trend of remaining almost unchanged diurnally throughout the year and over longtime variation, whereas the spatial extension of PWV is found to increase with time for the whole region. Investigations on the scale height and moist lapse rate computed from the radiosonde data signify the 

In [4]:
new_highlights = []
for text in highlights:
    text = text.strip().lower().replace('highlights', '').replace('\n',"").replace("•"," ")
    text = re.sub(r'\s\([^)]*\)\s', '', text)
    new_highlights.append(text.strip())

print(highlights[:5])
print(new_highlights[:5])

['Highlights\n•\nRadiative forcing by precipitable water vapour (PWV) estimated for tropical India.\n•\nPWV increase causes ≈0.0055–0.025 Wm−2 radiative forcing per year over long period.\n•\nVertical water vapour profiles, scale height and moist lapse rate are investigated.\n•\nThe significance of vertical temperature and water vapour profiles are realised.\n•\nPWV, similar to actual values are simulated to demonstrate these effects.', 'Highlights\n•\nThis study uses the microwave observations from emerging FY-3G/MWRI-RM.\n•\nThree machine learning algorithms are used to construct our models.\n•\nThe microwave observations can retrieve PWV not only over the ocean but also over land.', 'Highlights\n•\nThe Deep Belief Network (DBN) is utilized for downscaling SMAP soil moisture data.\n•\nThe DBN outperforms all other downscaling methods applied in this study.\n•\nThe performance of the DBN is thoroughly evaluated across various land cover types.\n•\nThe effectiveness of various input mo

In [11]:
link_to_keywords = {}

links = df['Link'].tolist()
for j, link in enumerate(links):
    keyword = keywords[j]
    keyword = re.sub(r'\s*\([^)]*\)\s*', '', keyword)
    new_text = ''
    if ";" in keyword:
        new_text = keyword
    else:
        for i, char in enumerate(keyword):
            try:
                if keyword[i].islower() and keyword[i+1].isupper():
                    new_text += char + ';'
                elif keyword[i].isupper() and keyword[i+1].isupper() and keyword[i+2].islower() and keyword[i+3]!=' ':
                    new_text += char + ';'
                else:
                    new_text += char
            except Exception as e:
                new_text += char

    items = new_text.lower().split(';')
    link_to_keywords[link] = items  
    # print(keyword, '\t', new_text, link)
    # if j > 10:
    #     break

In [6]:
new_df = pd.DataFrame()
new_df['Highlight'] = new_highlights
new_df['Abstract'] = new_abstracts
new_df['Link'] = links
new_df.to_excel("data/combine/all_expe_by_self_processed.xlsx", index=False)

In [12]:
import json

with open("data/combine/keywords.json", 'w') as f:
    json.dump(link_to_keywords, f, indent=4)

关键词数量统计

In [2]:
import pandas as pd
import json

df = pd.read_excel("data/combine/all_expe_by_self.xlsx")

link_to_keywords = {}
with open("data/combine/keywords.json", 'r') as f:
    link_to_keywords = json.load(f)

links = df['Link'].tolist()
j_titles = df['Title'].tolist()

In [9]:
title_to_keyword_num = {}

for i, link in enumerate(links):
    try:
        j_title = j_titles[i]
        keywords = link_to_keywords[link]
        if j_title not in title_to_keyword_num:
            title_to_keyword_num[j_title] = []
        title_to_keyword_num[j_title].append(len(keywords))
    except:
        continue

In [10]:
import numpy as np

for title in title_to_keyword_num:
    print(title, np.mean(title_to_keyword_num[title]))

Advances in Space Research 4.75
Applied Mathematical Modelling 5.05
Brain, Behavior, and Immunity 6.2
Chemical Engineering Journal 4.85
Complementary Therapies in Medicine 5.05
Composites Science and Technology 4.55
Ecological Indicators 4.85
Ecosystem Services 5.8
Human Factors in Healthcare 5.1
Journal of Economics and Business 4.7368421052631575
Journal of Theoretical Biology 5.0
Life Sciences 5.15
Social Networks 5.55
Pattern Recognition 4.05
Cell 5.45
Journal of Advanced Research 5.05
European Polymer Journal 4.6
Energy Storage Materials 4.8
Journal of Aging Studies 5.55
Journal of Financial Stability 4.4
Journal of Vocational Behavior 4.85
Veterinary Microbiology 4.7
The American Journal of Emergency Medicine 5.05
Applied Energy 5.3
Dental Materials 5.25
Carbon 4.85
Journal of Cleaner Production 5.1


In [12]:
for title in title_to_keyword_num:
    print(title, np.max(title_to_keyword_num[title]))

Advances in Space Research 7
Applied Mathematical Modelling 7
Brain, Behavior, and Immunity 12
Chemical Engineering Journal 6
Complementary Therapies in Medicine 8
Composites Science and Technology 6
Ecological Indicators 7
Ecosystem Services 9
Human Factors in Healthcare 7
Journal of Economics and Business 7
Journal of Theoretical Biology 11
Life Sciences 7
Social Networks 8
Pattern Recognition 6
Cell 7
Journal of Advanced Research 6
European Polymer Journal 6
Energy Storage Materials 6
Journal of Aging Studies 8
Journal of Financial Stability 6
Journal of Vocational Behavior 7
Veterinary Microbiology 7
The American Journal of Emergency Medicine 6
Applied Energy 7
Dental Materials 7
Carbon 6
Journal of Cleaner Production 6
